# LLM Context Window, Word Probability Correction, and MECW Detection

This notebook implements the methodology from the supplied research report as executable Python modules. It is intentionally deterministic: all model inspection is performed from logits and hidden states, with no stochastic sampler, no temperature sampling, and no top-k/top-p filtering.

The modules are:

1. Pimentel & Meister corrected word probabilities for BoW tokenizers.
2. Sampler-free confidence proxies from logits and hidden-state trajectories.
3. Dynamic Maximum Effective Context Window (MECW) detection from widening confidence intervals.
4. Domain saturation via n-gram perplexity and an elbow point.
5. SATLUTION-style L2 state management using a local `hypothesis.md` buffer inside `research`.

## Module 0: Runtime Setup

The report assumes a standard PyTorch plus HuggingFace stack. This first cell makes the notebook executable in a fresh environment by installing missing runtime packages, then imports the deterministic tooling used by every later module.

In [ ]:
# Auto-install only the packages required by the requested notebook.
# This cell intentionally does not install optional NLP stacks such as spaCy or NLTK;
# the n-gram pipeline below uses a lightweight regex tokenizer to stay portable.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "torch": "torch",
    "transformers": "transformers",
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}

missing_packages = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union
import json
import math
import random
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# Deterministic execution. No stochastic sampler is used anywhere in this notebook.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Resolve the research directory without touching files outside it.
def resolve_research_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == "research":
        return cwd
    candidate = cwd / "research"
    if candidate.exists():
        return candidate.resolve()
    raise RuntimeError("Launch this notebook from the repository root or from the research directory.")

RESEARCH_DIR = resolve_research_dir()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Research directory: {RESEARCH_DIR}")
print(f"Device: {DEVICE}")

def load_causal_lm(model_name: str = "gpt2", device: torch.device = DEVICE):
    """Load a small causal LM with hidden states enabled for structural metrics."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.to(device)
    model.eval()
    return tokenizer, model

## Module 1: Word Probability Correction (The Pimentel & Meister Fix)

The report highlights the marginalization error created by beginning-of-word (BoW) tokenizers such as GPT-style BPE tokenizers. A naive word probability multiplies the probabilities of the subword tokens that spell the word. For BoW tokenizers this is biased, because the model may reserve probability mass for starting a different next word rather than continuing the current word segmentation path.

For a word represented as a subword sequence `sw`, and for `S` as the BoW subset of the vocabulary, the corrected probability is:

`p(w | w_<t) = p(sw | sw_<t) * [sum_{s in S} p(s | sw_<t o sw)] / [sum_{s in S} p(s | sw_<t)]`.

The denominator measures how much probability mass the model assigns to starting a new word before the candidate word. The numerator measures the same BoW mass after the candidate word has been consumed. The ratio corrects the marginalization error while preserving deterministic logit inspection.

In [ ]:
_BOW_MASK_CACHE: Dict[Tuple[int, int, str], torch.Tensor] = {}

def _looks_like_bow_token(token: str, decoded: str, token_id: int, tokenizer) -> bool:
    """Heuristic BoW detector for common HF tokenizers.

    GPT-2 and RoBERTa byte-level BPE use 'Ġ'. SentencePiece often uses '▁'.
    Some tokenizers expose decoded tokens with a literal leading space. WordPiece
    marks continuation pieces with '##', so non-continuation tokens are a fallback
    only when explicit BoW markers are unavailable.
    """
    if token_id in set(getattr(tokenizer, "all_special_ids", [])):
        return False
    if token.startswith(("Ġ", "▁")):
        return True
    if decoded.startswith((" ", "\n", "\t")):
        return True
    return False

def get_bow_token_mask(tokenizer, vocab_size: Optional[int] = None, device: Optional[torch.device] = None) -> torch.Tensor:
    """Return a boolean mask over vocabulary ids that begin a new word."""
    vocab_size = int(vocab_size or len(tokenizer))
    device = device or torch.device("cpu")
    cache_key = (id(tokenizer), vocab_size, str(device))
    if cache_key in _BOW_MASK_CACHE:
        return _BOW_MASK_CACHE[cache_key]

    mask = torch.zeros(vocab_size, dtype=torch.bool, device=device)
    vocab = tokenizer.get_vocab()
    explicit_marker_count = 0
    fallback_wordpiece_ids: List[int] = []

    for token, token_id in vocab.items():
        if token_id >= vocab_size:
            continue
        try:
            decoded = tokenizer.convert_tokens_to_string([token])
        except Exception:
            decoded = token

        has_explicit_marker = token.startswith(("Ġ", "▁")) or decoded.startswith((" ", "\n", "\t"))
        if has_explicit_marker and token_id not in set(getattr(tokenizer, "all_special_ids", [])):
            mask[token_id] = True
            explicit_marker_count += 1
        elif not token.startswith("##") and token_id not in set(getattr(tokenizer, "all_special_ids", [])):
            fallback_wordpiece_ids.append(token_id)

    # If the tokenizer has no explicit BoW markers, use the WordPiece convention.
    if explicit_marker_count == 0 and fallback_wordpiece_ids:
        mask[torch.tensor(fallback_wordpiece_ids, dtype=torch.long, device=device)] = True

    _BOW_MASK_CACHE[cache_key] = mask
    return mask

def compute_corrected_word_probability(
    logits: torch.Tensor,
    token_ids: Union[Sequence[int], torch.Tensor],
    tokenizer,
    eps: float = 1e-9,
) -> Dict[str, torch.Tensor]:
    """Compute the Pimentel & Meister corrected word probability.

    Parameters
    ----------
    logits:
        Tensor shaped [steps, vocab] or [batch, steps, vocab]. For a target word
        split into k subword tokens, the first k rows must predict each subword
        token in order and row k must predict the next token after the full word.
        This gives exactly the denominator BoW mass before the word and the
        numerator BoW mass after the word.
    token_ids:
        The k subword token ids for the target word. A 1-D tensor is broadcast
        across the batch; a 2-D tensor must be [batch, k].
    tokenizer:
        HuggingFace tokenizer used to build the BoW vocabulary mask.
    eps:
        Numerical guard added to the denominator.
    """
    if logits.ndim == 2:
        logits = logits.unsqueeze(0)
    if logits.ndim != 3:
        raise ValueError("logits must have shape [steps, vocab] or [batch, steps, vocab]")

    batch_size, steps, vocab_size = logits.shape
    ids = torch.as_tensor(token_ids, dtype=torch.long, device=logits.device)
    if ids.ndim == 1:
        ids = ids.unsqueeze(0).expand(batch_size, -1)
    if ids.ndim != 2 or ids.shape[0] != batch_size:
        raise ValueError("token_ids must be [k] or [batch, k]")

    k = ids.shape[1]
    if steps < k + 1:
        raise ValueError(
            f"Need at least k+1={k + 1} logit rows: k rows for subwords and one row after the word."
        )

    probs = torch.softmax(logits[:, : k + 1, :], dim=-1)
    subword_probs = probs[:, :k, :].gather(dim=-1, index=ids.unsqueeze(-1)).squeeze(-1)
    p_subword_sequence = subword_probs.prod(dim=-1)

    bow_mask = get_bow_token_mask(tokenizer, vocab_size=vocab_size, device=logits.device)
    if bow_mask.sum().item() == 0:
        raise ValueError("No BoW tokens were detected for this tokenizer vocabulary.")

    bow_mass_before = probs[:, 0, :][:, bow_mask].sum(dim=-1)
    bow_mass_after = probs[:, k, :][:, bow_mask].sum(dim=-1)
    correction_factor = bow_mass_after / (bow_mass_before + eps)
    corrected_probability = p_subword_sequence * correction_factor
    corrected_probability = torch.nan_to_num(corrected_probability, nan=0.0, posinf=1.0, neginf=0.0).clamp_min(0.0)

    return {
        "corrected_probability": corrected_probability,
        "base_subword_probability": p_subword_sequence,
        "subword_probabilities": subword_probs,
        "bow_mass_before": bow_mass_before,
        "bow_mass_after": bow_mass_after,
        "correction_factor": correction_factor,
        "bow_token_count": bow_mask.sum().detach().cpu(),
    }

def get_aligned_logits_for_word(
    model,
    tokenizer,
    context_text: str,
    target_word_text: str,
    device: torch.device = DEVICE,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Create the k+1 aligned logit rows required by compute_corrected_word_probability."""
    context_ids = tokenizer(context_text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    target_ids = tokenizer(target_word_text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    if context_ids.numel() == 0:
        raise ValueError("context_text must tokenize to at least one token")
    if target_ids.numel() == 0:
        raise ValueError("target_word_text must tokenize to at least one token")

    max_positions = int(getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", 1024)))
    max_context_tokens = max_positions - target_ids.shape[1]
    if max_context_tokens < 1:
        raise ValueError("target word is too long for the model position limit")
    if context_ids.shape[1] > max_context_tokens:
        context_ids = context_ids[:, -max_context_tokens:]

    input_ids = torch.cat([context_ids, target_ids], dim=1)
    with torch.no_grad():
        outputs = model(input_ids=input_ids)

    c = context_ids.shape[1]
    k = target_ids.shape[1]
    positions = torch.arange(c - 1, c + k, device=device)
    aligned_logits = outputs.logits[0, positions, :]
    return aligned_logits, target_ids[0]

## Module 2: Confidence Intervals and MECW Detection (Sampler-Free)

The report rejects sampler-derived confidence because stochastic decoding changes the future context after every random draw. It also warns that raw logits are overconfident under the LLM Perception Tunnel: high token probability may reflect syntactic predictability rather than semantic certainty.

Without a trained calibration probe, this notebook implements a deterministic proxy aligned with the report:

- logit entropy and top-1 margin expose distributional uncertainty;
- hidden-state interquartile range (IQR) approximates quantile-regression interval width;
- hidden trajectory variation and inter-layer variance approximate Structural Confidence.

The resulting `ci_width_proxy` is not a calibrated empirical confidence interval. It is a sampler-free structural width signal suitable for MECW monitoring.

In [ ]:
def _as_batch_logits(logits: torch.Tensor) -> torch.Tensor:
    if logits.ndim == 2:
        return logits.unsqueeze(0)
    if logits.ndim == 3:
        return logits
    raise ValueError("logits must be [seq, vocab] or [batch, seq, vocab]")

def _unit_scale(x: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """Scale each batch row to [0, 1] while preserving zero tensors."""
    if x.ndim == 1:
        x = x.unsqueeze(0)
    xmin = x.amin(dim=-1, keepdim=True)
    xmax = x.amax(dim=-1, keepdim=True)
    return (x - xmin) / (xmax - xmin + eps)

def extract_confidence_metrics(
    logits: torch.Tensor,
    hidden_states: Optional[Union[Tuple[torch.Tensor, ...], List[torch.Tensor], torch.Tensor]],
    eps: float = 1e-9,
) -> Dict[str, Any]:
    """Extract sampler-free pseudo-confidence metrics from logits and hidden states."""
    logits = _as_batch_logits(logits).float()
    probs = torch.softmax(logits, dim=-1)
    vocab_size = logits.shape[-1]

    entropy = -(probs * torch.log(probs + eps)).sum(dim=-1)
    normalized_entropy = entropy / math.log(max(vocab_size, 2))
    top2 = torch.topk(probs, k=2, dim=-1).values
    top1_margin = (top2[..., 0] - top2[..., 1]).clamp_min(0.0)
    logit_iqr = torch.quantile(logits, 0.75, dim=-1) - torch.quantile(logits, 0.25, dim=-1)

    if hidden_states is None:
        hidden_iqr = torch.zeros_like(entropy)
        trajectory_energy = torch.zeros_like(entropy)
        layer_variance = torch.zeros_like(entropy)
    else:
        if isinstance(hidden_states, torch.Tensor):
            final_hidden = hidden_states.float()
            if final_hidden.ndim == 2:
                final_hidden = final_hidden.unsqueeze(0)
            layer_variance = torch.zeros(final_hidden.shape[:2], device=final_hidden.device)
        else:
            selected_layers = [h.float() for h in hidden_states[-min(4, len(hidden_states)):]]
            final_hidden = selected_layers[-1]
            layer_stack = torch.stack(selected_layers, dim=0)
            layer_variance = layer_stack.var(dim=0, unbiased=False).mean(dim=-1)

        hidden_iqr = torch.quantile(final_hidden, 0.75, dim=-1) - torch.quantile(final_hidden, 0.25, dim=-1)
        if final_hidden.shape[1] > 1:
            deltas = final_hidden[:, 1:, :] - final_hidden[:, :-1, :]
            trajectory_energy = torch.linalg.vector_norm(deltas, dim=-1)
            trajectory_energy = F.pad(trajectory_energy, pad=(1, 0), value=0.0)
        else:
            trajectory_energy = torch.zeros(final_hidden.shape[:2], device=final_hidden.device)

    # Combine uncertainty signals. High entropy, low margin, wide activation IQR,
    # high trajectory energy, and high inter-layer variance all widen the proxy CI.
    token_ci_width = (
        normalized_entropy
        + (1.0 - top1_margin.clamp(0.0, 1.0))
        + _unit_scale(hidden_iqr, eps=eps)
        + _unit_scale(trajectory_energy, eps=eps)
        + _unit_scale(layer_variance, eps=eps)
    ) / 5.0

    return {
        "entropy": entropy.detach(),
        "normalized_entropy": normalized_entropy.detach(),
        "top1_margin": top1_margin.detach(),
        "logit_iqr": logit_iqr.detach(),
        "hidden_iqr": hidden_iqr.detach(),
        "trajectory_energy": trajectory_energy.detach(),
        "layer_variance": layer_variance.detach(),
        "token_ci_width": token_ci_width.detach(),
        "ci_width_proxy": float(token_ci_width.mean().detach().cpu()),
        "mean_entropy": float(entropy.mean().detach().cpu()),
        "mean_top1_margin": float(top1_margin.mean().detach().cpu()),
        "mean_hidden_iqr": float(hidden_iqr.mean().detach().cpu()),
        "mean_trajectory_energy": float(trajectory_energy.mean().detach().cpu()),
    }

## Module 3: Dynamic MECW Detection

The report defines Maximum Effective Context Window (MECW) as task-dependent and empirical: it is the longest context length before additional context causes measurable degradation, such as sharply widening confidence intervals. This module simulates an expanding context and monitors the moving width of the structural confidence proxy. A breach occurs when the variance of the moving width exceeds `epsilon`.

In [ ]:
def _coerce_context_tokens(context_tokens: Union[str, Sequence[int], torch.Tensor], tokenizer, device: torch.device) -> torch.Tensor:
    if isinstance(context_tokens, str):
        ids = tokenizer(context_tokens, return_tensors="pt", add_special_tokens=False).input_ids[0]
    else:
        ids = torch.as_tensor(context_tokens, dtype=torch.long)
        if ids.ndim == 2:
            ids = ids[0]
    return ids.to(device)

def detect_mecw(
    context_tokens: Union[str, Sequence[int], torch.Tensor],
    window_step: int,
    model,
    tokenizer,
    epsilon: float = 2e-4,
    min_windows: int = 4,
    moving_average_span: int = 3,
    device: torch.device = DEVICE,
    return_details: bool = False,
) -> Union[bool, Dict[str, Any]]:
    """Detect MECW breach by expanding context and tracking CI-width variance."""
    if window_step <= 0:
        raise ValueError("window_step must be positive")
    if min_windows < 2:
        raise ValueError("min_windows must be at least 2")

    ids = _coerce_context_tokens(context_tokens, tokenizer, device=device)
    max_positions = int(getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", 1024)))
    ids = ids[:max_positions]

    lengths: List[int] = []
    widths: List[float] = []
    moving_widths: List[float] = []
    moving_variances: List[float] = []
    breach_index: Optional[int] = None

    upper_bounds = list(range(window_step, int(ids.numel()) + 1, window_step))
    if upper_bounds and upper_bounds[-1] != int(ids.numel()):
        upper_bounds.append(int(ids.numel()))
    elif not upper_bounds and ids.numel() > 0:
        upper_bounds = [int(ids.numel())]

    for end in upper_bounds:
        input_ids = ids[:end].unsqueeze(0)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, output_hidden_states=True)
        metrics = extract_confidence_metrics(outputs.logits, outputs.hidden_states)
        width = float(metrics["ci_width_proxy"])
        lengths.append(end)
        widths.append(width)

        recent_widths = widths[-moving_average_span:]
        moving_widths.append(float(np.mean(recent_widths)))
        if len(moving_widths) >= min_windows:
            variance = float(np.var(moving_widths[-min_windows:]))
            moving_variances.append(variance)
            if breach_index is None and variance > epsilon:
                breach_index = end
        else:
            moving_variances.append(0.0)

    details = {
        "breached": breach_index is not None,
        "breach_index": breach_index,
        "epsilon": epsilon,
        "window_step": window_step,
        "lengths": lengths,
        "ci_widths": widths,
        "moving_average_widths": moving_widths,
        "moving_variances": moving_variances,
        "max_variance": max(moving_variances) if moving_variances else 0.0,
    }
    return details if return_details else bool(details["breached"])

## Module 4: N-Gram Perplexity and the Elbow Method for Domain Saturation

The report parallels n-gram perplexity with the K-means elbow method. Domain text is tokenized, stop words are filtered, and n-grams are treated as compact domain structure units. As more domain context is added, model perplexity should initially drop; once the domain structure is saturated, the curve flattens. The elbow is the maximum curvature point, computed here through the second derivative of the perplexity-vs-context curve.

For strict alignment with Module 1, the default perplexity path uses corrected word probabilities. A faster token-level loss path is also included for large experiments.

In [ ]:
DEFAULT_STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "he", "in",
    "is", "it", "its", "of", "on", "or", "that", "the", "to", "was", "were", "will", "with",
    "this", "these", "those", "into", "than", "then", "when", "where", "while", "without",
}

TOKEN_PATTERN = re.compile(r"[A-Za-z][A-Za-z0-9_'-]*|[0-9]+(?:\.[0-9]+)?")

def build_ngram_domain_pipeline(
    domain_strings: Sequence[str],
    n: int = 3,
    stop_words: Optional[Iterable[str]] = None,
) -> Dict[str, Any]:
    """Tokenize domain strings, filter stop words, and build unique n-grams."""
    if n <= 0:
        raise ValueError("n must be positive")
    stop = set(DEFAULT_STOP_WORDS if stop_words is None else stop_words)

    raw_tokens: List[str] = []
    for text in domain_strings:
        raw_tokens.extend(match.group(0).lower() for match in TOKEN_PATTERN.finditer(text))

    filtered_tokens = [tok for tok in raw_tokens if tok not in stop and len(tok) > 1]
    ngrams = [tuple(filtered_tokens[i : i + n]) for i in range(max(0, len(filtered_tokens) - n + 1))]

    seen = set()
    unique_ngrams: List[Tuple[str, ...]] = []
    for gram in ngrams:
        if gram not in seen:
            seen.add(gram)
            unique_ngrams.append(gram)

    return {
        "raw_tokens": raw_tokens,
        "filtered_tokens": filtered_tokens,
        "ngrams": ngrams,
        "unique_ngrams": unique_ngrams,
        "ngram_texts": [" ".join(gram) for gram in unique_ngrams],
        "n": n,
    }

def corrected_word_cross_entropy(
    text: str,
    model,
    tokenizer,
    device: torch.device = DEVICE,
    max_words: int = 32,
    eps: float = 1e-9,
) -> float:
    """Average -log p(w) using the Pimentel & Meister correction."""
    words = re.findall(r"\S+", text)
    words = words[:max_words]
    if len(words) < 2:
        return float("nan")

    losses: List[float] = []
    context_text = words[0]
    for word in words[1:]:
        # GPT-style BoW tokenizers represent most non-initial words with a leading space.
        target_text = " " + word
        aligned_logits, target_ids = get_aligned_logits_for_word(model, tokenizer, context_text, target_text, device=device)
        result = compute_corrected_word_probability(aligned_logits, target_ids, tokenizer, eps=eps)
        probability = float(result["corrected_probability"].reshape(-1)[0].detach().cpu())
        losses.append(-math.log(max(probability, eps)))
        context_text = context_text + " " + word

    return float(np.mean(losses)) if losses else float("nan")

def token_cross_entropy(
    text: str,
    model,
    tokenizer,
    device: torch.device = DEVICE,
) -> float:
    """Fast token-level cross entropy fallback for long curves."""
    max_positions = int(getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", 1024)))
    input_ids = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_positions).input_ids.to(device)
    if input_ids.shape[1] < 2:
        return float("nan")
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    shift_logits = outputs.logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="mean")
    return float(loss.detach().cpu())

def calculate_context_perplexities(
    text_chunks: Sequence[str],
    model,
    tokenizer,
    lengths: Sequence[int],
    device: torch.device = DEVICE,
    use_corrected_words: bool = True,
    max_words_per_eval: int = 32,
) -> Dict[str, List[float]]:
    """Calculate average cross entropy and perplexity for increasing context sizes."""
    if not text_chunks:
        raise ValueError("text_chunks must not be empty")

    clean_lengths = sorted({int(length) for length in lengths if int(length) > 0})
    evaluated_lengths: List[int] = []
    cross_entropies: List[float] = []
    perplexities: List[float] = []

    for length in clean_lengths:
        selected = list(text_chunks[: min(length, len(text_chunks))])
        context_text = " ".join(selected)
        if use_corrected_words:
            ce = corrected_word_cross_entropy(context_text, model, tokenizer, device=device, max_words=max_words_per_eval)
        else:
            ce = token_cross_entropy(context_text, model, tokenizer, device=device)

        ppl = float(math.exp(min(ce, 50.0))) if math.isfinite(ce) else float("nan")
        evaluated_lengths.append(length)
        cross_entropies.append(float(ce))
        perplexities.append(ppl)

    return {
        "lengths": evaluated_lengths,
        "cross_entropies": cross_entropies,
        "perplexities": perplexities,
        "mode": "corrected_word" if use_corrected_words else "token",
    }

def find_elbow_point(perplexities: Sequence[float], lengths: Sequence[int]) -> Dict[str, Any]:
    """Find maximum curvature point using numerical gradients."""
    x = np.asarray(lengths, dtype=float)
    y = np.asarray(perplexities, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) == 0:
        raise ValueError("No finite points are available for elbow detection")
    if len(x) < 3:
        return {"elbow_index": int(len(x) - 1), "elbow_length": int(x[-1]), "curvature": [0.0] * len(x)}

    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-9)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-9)
    dy = np.gradient(y_norm, x_norm)
    ddy = np.gradient(dy, x_norm)
    curvature = np.abs(ddy) / np.power(1.0 + dy ** 2, 1.5)
    interior = curvature.copy()
    interior[0] = -np.inf
    interior[-1] = -np.inf
    elbow_index = int(np.argmax(interior))
    return {
        "elbow_index": elbow_index,
        "elbow_length": int(x[elbow_index]),
        "elbow_perplexity": float(y[elbow_index]),
        "curvature": curvature.tolist(),
    }

def plot_perplexity_elbow(perplexity_result: Dict[str, List[float]], elbow: Dict[str, Any]):
    lengths = perplexity_result["lengths"]
    perplexities = perplexity_result["perplexities"]
    plt.figure(figsize=(9, 5))
    plt.plot(lengths, perplexities, marker="o", linewidth=2, label="Perplexity")
    plt.axvline(elbow["elbow_length"], linestyle="--", color="red", label=f"Elbow = {elbow['elbow_length']}")
    plt.xlabel("Number of domain n-gram chunks")
    plt.ylabel("Perplexity")
    plt.title("Domain Saturation via N-Gram Perplexity Elbow")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Module 5: SATLUTION Agentic State Management (`hypothesis.md` Pattern)

The report maps MECW control to SATLUTION/Kairos-style agent memory. L0 contains immutable control rules, L1 stores project-specific conventions, and L2 is a temporary session context. When MECW is breached, the agent should not keep forcing more content into the active context. It should halt, compress the active state into an external buffer such as `hypothesis.md`, flush the active context, and restart from a clean window.

The class below implements that L2 behavior. It writes only to `research/hypothesis.md` by default.

In [ ]:
def ensure_path_inside_research(path: Union[str, Path]) -> Path:
    resolved = Path(path).resolve()
    research = RESEARCH_DIR.resolve()
    try:
        resolved.relative_to(research)
    except ValueError as exc:
        raise ValueError(f"Refusing to write outside research directory: {resolved}") from exc
    return resolved

@dataclass
class AgentContextManager:
    model: Any
    tokenizer: Any
    hypothesis_path: Union[str, Path] = field(default_factory=lambda: RESEARCH_DIR / "hypothesis.md")
    window_step: int = 32
    epsilon: float = 1e-8
    device: torch.device = DEVICE
    active_context_tokens: List[int] = field(default_factory=list)
    active_notes: List[str] = field(default_factory=list)

    def __post_init__(self):
        self.hypothesis_path = ensure_path_inside_research(self.hypothesis_path)

    def add_text(self, text: str) -> None:
        token_ids = self.tokenizer(text, add_special_tokens=False).input_ids
        self.active_context_tokens.extend(int(tok) for tok in token_ids)
        self.active_notes.append(text)

    def summarize_active_context(self, max_tokens: int = 80) -> Dict[str, Any]:
        head_ids = self.active_context_tokens[: max_tokens // 2]
        tail_ids = self.active_context_tokens[-max_tokens // 2 :] if self.active_context_tokens else []
        summary_ids = head_ids + ([] if len(self.active_context_tokens) <= max_tokens else tail_ids)
        decoded_summary = self.tokenizer.decode(summary_ids, skip_special_tokens=True)
        return {
            "token_count": len(self.active_context_tokens),
            "note_count": len(self.active_notes),
            "summary": decoded_summary[:1200],
            "recent_note": self.active_notes[-1][:600] if self.active_notes else "",
        }

    def append_hypothesis_record(self, record: Dict[str, Any]) -> None:
        self.hypothesis_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.hypothesis_path.exists():
            self.hypothesis_path.write_text("# L2 Hypothesis Buffer\n\n", encoding="utf-8")
        payload = json.dumps(record, ensure_ascii=False, indent=2)
        with self.hypothesis_path.open("a", encoding="utf-8") as handle:
            handle.write("\n## MECW Context Flush\n\n")
            handle.write("```json\n")
            handle.write(payload)
            handle.write("\n```\n")

    def reset_active_context(self) -> None:
        self.active_context_tokens.clear()
        self.active_notes.clear()

    def monitor_and_flush_if_needed(self) -> Dict[str, Any]:
        if not self.active_context_tokens:
            return {"flushed": False, "reason": "active context is empty"}

        mecw_details = detect_mecw(
            self.active_context_tokens,
            window_step=self.window_step,
            model=self.model,
            tokenizer=self.tokenizer,
            epsilon=self.epsilon,
            device=self.device,
            return_details=True,
        )

        if not mecw_details["breached"]:
            return {"flushed": False, "mecw": mecw_details}

        record = {
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "event": "MECW_BREACH_CONTEXT_FLUSH",
            "mecw": mecw_details,
            "compressed_l2_state": self.summarize_active_context(),
            "next_action": "Start a clean active context and rehydrate only the compressed L2 summary.",
        }
        self.append_hypothesis_record(record)
        self.reset_active_context()
        return {"flushed": True, "record": record}

## End-to-End Execution

The `main()` block loads GPT-2, runs dummy domain data through all modules, validates the BoW mask and corrected probability calculation, plots the perplexity elbow, and exercises the SATLUTION-style context manager. The MECW threshold in the demo is intentionally sensitive so that the flush path is observable on short local examples.

In [ ]:
def main():
    tokenizer, model = load_causal_lm("gpt2", device=DEVICE)

    # Module 1 demo: corrected probability for one BoW word.
    context_text = "The deterministic language model estimates"
    target_word = " uncertainty"
    aligned_logits, target_ids = get_aligned_logits_for_word(model, tokenizer, context_text, target_word, device=DEVICE)
    corrected = compute_corrected_word_probability(aligned_logits, target_ids, tokenizer)
    corrected_p = corrected["corrected_probability"].reshape(-1)[0]
    assert torch.isfinite(corrected_p), "corrected probability must be finite"
    assert corrected_p.item() >= 0.0, "corrected probability must be non-negative"
    assert int(corrected["bow_token_count"]) > 0, "BoW mask must contain GPT-style BoW tokens"

    print("Module 1: corrected word probability")
    print({k: (float(v.reshape(-1)[0].detach().cpu()) if torch.is_tensor(v) and v.numel() else v) for k, v in corrected.items() if k != "subword_probabilities"})

    # Module 2 demo: structural confidence metrics.
    sample_text = "Context rot begins when confidence intervals widen under excess domain tokens."
    input_ids = tokenizer(sample_text, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, output_hidden_states=True)
    confidence = extract_confidence_metrics(outputs.logits, outputs.hidden_states)
    for key in ["ci_width_proxy", "mean_entropy", "mean_top1_margin", "mean_hidden_iqr", "mean_trajectory_energy"]:
        assert math.isfinite(float(confidence[key])), f"{key} must be finite"
    print("\nModule 2: confidence metrics")
    print({key: confidence[key] for key in ["ci_width_proxy", "mean_entropy", "mean_top1_margin", "mean_hidden_iqr", "mean_trajectory_energy"]})

    # Module 3 demo: MECW detection on an expanding synthetic context.
    long_context = " ".join([
        "deterministic logits hidden states structural confidence ngram perplexity elbow saturation",
        "agent memory context rot mecw satlution hypothesis buffer compile test analyze evolve",
        "domain adaptation corrected word probability bow tokenizer marginalization surprisal",
    ] * 8)
    context_tokens = tokenizer(long_context, add_special_tokens=False).input_ids
    mecw = detect_mecw(context_tokens, window_step=24, model=model, tokenizer=tokenizer, epsilon=1e-10, min_windows=3, device=DEVICE, return_details=True)
    assert isinstance(mecw["breached"], bool), "MECW breached flag must be boolean"
    print("\nModule 3: MECW details")
    print({k: mecw[k] for k in ["breached", "breach_index", "max_variance", "epsilon"]})

    # Module 4 demo: domain n-grams, corrected-word perplexity curve, and elbow plot.
    domain_strings = [
        "Corrected word probability fixes BoW tokenizer marginalization and stabilizes surprisal.",
        "Structural confidence uses hidden-state trajectories, activation IQR, and logit entropy.",
        "MECW detection monitors widening intervals as context noise accumulates.",
        "SATLUTION stores compressed hypotheses in an L2 markdown buffer before resetting context.",
        "N-gram perplexity falls quickly, then reaches an elbow when domain structure saturates.",
    ]
    pipeline = build_ngram_domain_pipeline(domain_strings, n=3)
    max_len = min(10, len(pipeline["ngram_texts"]))
    lengths = list(range(2, max_len + 1, 2)) or [1]
    perplexity_result = calculate_context_perplexities(
        pipeline["ngram_texts"],
        model=model,
        tokenizer=tokenizer,
        lengths=lengths,
        device=DEVICE,
        use_corrected_words=True,
        max_words_per_eval=24,
    )
    elbow = find_elbow_point(perplexity_result["perplexities"], perplexity_result["lengths"])
    print("\nModule 4: n-gram pipeline and elbow")
    print({"unique_ngrams": len(pipeline["unique_ngrams"]), "perplexities": perplexity_result["perplexities"], "elbow": elbow})
    plot_perplexity_elbow(perplexity_result, elbow)

    # Module 5 demo: SATLUTION/Kairos L2 context flush.
    manager = AgentContextManager(model=model, tokenizer=tokenizer, window_step=24, epsilon=1e-10, device=DEVICE)
    manager.add_text(long_context)
    flush_result = manager.monitor_and_flush_if_needed()
    print("\nModule 5: AgentContextManager flush result")
    print({"flushed": flush_result.get("flushed"), "hypothesis_path": str(manager.hypothesis_path)})

    return {
        "corrected_probability": float(corrected_p.detach().cpu()),
        "confidence": {key: confidence[key] for key in ["ci_width_proxy", "mean_entropy", "mean_top1_margin"]},
        "mecw": mecw,
        "perplexity": perplexity_result,
        "elbow": elbow,
        "flush_result": flush_result,
    }

if __name__ == "__main__":
    results = main()